# Création de la base d'apprentissage

## 1. Présentation générale

Ce notebook constitue l'étape cruciale de notre projet : la **réconciliation de données multi-sources** pour bâtir notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives.

Nous centralisons alors dans un premier temps des fichiers issus de l'API Soccerdata, regroupant ainsi des données FBref et des données Understat. De plus, nous centralisons également des données Transfermarkt (les données financières ainsi que notre variable cible : la valeur marchande) et du mapping issu de worldfootballR.

Nous réalisons ensuite une fusion à plusieurs niveaux : nous utilisons les identifiants connus des joueurs puisdu fuzzy-mapping.


Nous devrions obtenir finalement une table prête pour de plus profondes analyses voire pour de la modélisation, mêlant ainsi des données issues des performances sportives à des données analysant la valeur marchande des joueurs de football.

Pour ce faire, nous importons dans un premier temps des packages et des fonctions nécessaires à la création de notre base d'apprentissage.

In [9]:
# Importation des packages nécessaires

import pandas as pd
import os
import sys

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from merging import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Chargement et préparation des sources


Nous importons dans un premier temps nos trois fichiers comprenant nos données :
- issues du mapping de worldfootballR
- issues de transfermarkt
- issues de soccerdata

In [10]:
# Chargement des données du mapping
df_mapping_initial = pd.read_csv("../data_finale/mapping_worldfootballR/mapping_fbref_tm.csv", encoding='latin1')

# Chargement des données du dataset Soccerdata
df_soccerdata_initial = pd.read_csv("../data/soccerdata/data_final_soccerdata.csv")

# Chargement des données du dataset Transfermarkt
df_players = pd.read_csv("../data/transfermarkt_datasets/players.csv")
df_valuations = pd.read_csv("../data/transfermarkt_datasets/player_valuations.csv")

# Préparation des données de Transfermarkt
df_tm_initial = prepare_transfermarkt_data(
    df_players,
    df_valuations
)

# Chargement des données du dataset de blessures Transfermarkt
df_blessures = pd.read_csv("../data/dataset_blessures.csv")

# Préparation des données de blessures Transfermarkt
df_blessures_initial = aggregate_injuries_by_season(df_blessures)

Plutôt que de traiter chaque dataframe manuellement ici, nous utilisons la fonction match_player_data. Cette fonction encapsule toute la logique de nettoyage définie précédemment :

- Correction de l'encoding : Application de fix_encoding sur les noms FBref.

- Normalisation des noms : Suppression des accents, mise en minuscule et nettoyage des caractères spéciaux via normalize_name.

- Harmonisation des dates : Extraction de l'année de naissance (dob_key) pour faciliter le matching entre les sources.

- Création de clés composites : Génération de clés basées sur "Prénom + Nom" pour Transfermarkt.

In [11]:
# Nous appliquons les logiques décrites ci-dessus
df_mapping, df_soccerdata, df_tm, df_blessures = match_player_data(df_mapping_initial, df_soccerdata_initial,
                                                      df_tm_initial, df_blessures_initial)

## 3. La fusion à multi-niveaux

Nous appliquons ensuite une stratégie de fusion des bases de données en 2 étapes pour maximiser le taux de correspondance.

Dans un premier temps, nous réalisons une jointure exacte via le dictionnaire de mapping.

Ensuite, nous effectuons une recherche plus floue (fuzzy) sur le mapping avec un seuil supérieur à 90%.

In [12]:
df_final, still_missing = run_player_matching(df_soccerdata, df_mapping, df_tm, df_blessures)

[1] Nom exact (mapping)     : 16280 | restants : 833
[2] Fuzzy nom (mapping)     :   283 | restants : 550
[3.1] Match direct TM (Exact) :    71 | restants : 479
[3.2] Match direct TM (Fuzzy) :    34 | restants : 439


Nous pouvons enfin importer notre base d'apprentissage sous le format CSV.

In [15]:
df_final.to_csv(r'..\data_finale\base_apprentissage.csv', index=False, sep=',', encoding='utf-8-sig')
still_missing.to_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', index=False, sep=',', encoding='utf-8-sig')
df_soccerdata.to_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

## **Les joueurs orphelins**

In [16]:
still_missing = pd.read_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', encoding='utf-8-sig')
df_soccerdata = pd.read_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', encoding='utf-8-sig')

In [17]:
nb_joueurs_orphelins = still_missing['join_key'].nunique()
nb_joueurs_total = df_soccerdata['join_key'].nunique()

print(f"Joueurs orphelins : {nb_joueurs_orphelins}")
print(f"Taux joueurs orphelins : {nb_joueurs_orphelins / nb_joueurs_total:.2%}")

Joueurs orphelins : 422
Taux joueurs orphelins : 6.81%


In [18]:
orphelins_par_saison = (
    still_missing
    .groupby('season_year')
    .size()
    .sort_index()
)

print(orphelins_par_saison)

season_year
2020      1
2021     17
2022     25
2023     41
2024     16
2025    339
dtype: int64


In [19]:
temp_data = still_missing[still_missing['season_year']<2025]
temp_data

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
0,ENG-Premier League,2021,Manchester Utd,Will Fish,ENG,DF,17,2003.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,will fish,2003.0,2020
1,ENG-Premier League,2223,Brighton,Cameron Peupion,AUS,MF,19,2002.0,1,0,...,NaN,0,0.053613,0.000000,0.053613,0.000000,0.000000,cameron peupion,2002.0,2022
2,ENG-Premier League,2223,Manchester City,Shea Charles,NIR,"DF,MF",18,2003.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.129427,0.129427,shea charles,2003.0,2022
3,ENG-Premier League,2223,Southampton,Kami Doyle,ENG,MF,16,2005.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,kami doyle,2005.0,2022
4,ENG-Premier League,2223,Tottenham Hotspur,George Abbott,ENG,MF,16,2005.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,george abbott,2005.0,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,ITA-Serie A,2324,Salernitana,Mateusz Łęgowski,POL,MF,20,2003.0,29,10,...,NaN,0,NaN,NaN,NaN,NaN,NaN,mateusz u0141 u0119gowski,2003.0,2023
376,ITA-Serie A,2324,Udinese,Antonio Tikvić,CRO,DF,19,2004.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,antonio tikvi u0107,2004.0,2023
377,ITA-Serie A,2425,Lecce,Filip Marchwiński,POL,MF,22,2002.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,filip marchwi u0144ski,2002.0,2024
378,ITA-Serie A,2425,Torino,Alieu Njie,SWE,FW,19,2005.0,16,0,...,NaN,0,1.235164,0.560247,1.235164,1.752160,0.031047,alieu njie,2005.0,2024


In [20]:
df_soccerdata[df_soccerdata["player"] == "Danilo Santos"]

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
1498,ENG-Premier League,2223,Nottingham Forest,Danilo Santos,BRA,MF,21,2001.0,13,12,...,NaN,0,NaN,NaN,NaN,NaN,NaN,danilo santos,2001.0,2022
2085,ENG-Premier League,2324,Nottingham Forest,Danilo Santos,BRA,MF,22,2001.0,29,20,...,NaN,0,NaN,NaN,NaN,NaN,NaN,danilo santos,2001.0,2023
2655,ENG-Premier League,2425,Nottingham Forest,Danilo Santos,BRA,MF,23,2001.0,8,5,...,NaN,0,NaN,NaN,NaN,NaN,NaN,danilo santos,2001.0,2024


In [21]:
df_mapping[df_mapping["PlayerFBref"].str.contains("Danilo")]

,PlayerFBref,fbref_id,tm_id,TmPos,join_key
3067,Danilo,94b2001f,145707.0,Right-Back,danilo
3068,Danilo,315cdad5,519731.0,Centre-Forward,danilo
3070,Danilo,a816dbfb,808509.0,Defensive Midfield,danilo
3071,Danilo,d0f80d85,738498.0,Centre-Back,danilo
3072,Danilo,415a34c5,870414.0,Centre-Back,danilo
3073,Danilo Acosta,41f720a5,375074.0,Left-Back,danilo acosta
3074,Danilo Avelar,eacb45fd,140748.0,Centre-Back,danilo avelar
3075,Danilo Barbosa,0557ff4f,273438.0,Defensive Midfield,danilo barbosa
3076,Danilo Barcelos,26f4ef39,271666.0,Left-Back,danilo barcelos
3077,Danilo Batista,7c5fee43,1008105.0,Midfield,danilo batista


In [22]:
df_tm[df_tm["name"] == "Danilo"]

,player_id,valuation_season_year,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,...,url,current_club_domestic_competition_id,current_club_name,highest_market_value_in_eur,date,market_value_in_eur,player_club_domestic_competition_id,join_key,join_key_full,dob_key
444,32816,2019.0,NaN,Danilo,Danilo,2020,1025,danilo,Brazil,São Bernardo do Campo,...,https://www.transfermarkt.co.uk/danilo/profil/...,IT1,Bologna Football Club 1909,7500000.0,2020-08-25,600000.0,IT1,danilo,danilo,1984
445,32816,2020.0,NaN,Danilo,Danilo,2020,1025,danilo,Brazil,São Bernardo do Campo,...,https://www.transfermarkt.co.uk/danilo/profil/...,IT1,Bologna Football Club 1909,7500000.0,2020-12-29,600000.0,IT1,danilo,danilo,1984
4202,145707,2019.0,NaN,Danilo,Danilo,2025,614,danilo,Brazil,Bicas,...,https://www.transfermarkt.co.uk/danilo/profil/...,BRA1,Clube de Regatas do Flamengo,25000000.0,2020-08-25,15000000.0,BRA1,danilo,danilo,1991
4203,145707,2020.0,NaN,Danilo,Danilo,2025,614,danilo,Brazil,Bicas,...,https://www.transfermarkt.co.uk/danilo/profil/...,BRA1,Clube de Regatas do Flamengo,25000000.0,2021-06-03,20000000.0,BRA1,danilo,danilo,1991
4204,145707,2021.0,NaN,Danilo,Danilo,2025,614,danilo,Brazil,Bicas,...,https://www.transfermarkt.co.uk/danilo/profil/...,BRA1,Clube de Regatas do Flamengo,25000000.0,2022-06-07,13000000.0,BRA1,danilo,danilo,1991
4205,145707,2022.0,NaN,Danilo,Danilo,2025,614,danilo,Brazil,Bicas,...,https://www.transfermarkt.co.uk/danilo/profil/...,BRA1,Clube de Regatas do Flamengo,25000000.0,2023-06-15,15000000.0,BRA1,danilo,danilo,1991
4206,145707,2023.0,NaN,Danilo,Danilo,2025,614,danilo,Brazil,Bicas,...,https://www.transfermarkt.co.uk/danilo/profil/...,BRA1,Clube de Regatas do Flamengo,25000000.0,2024-06-05,10000000.0,BRA1,danilo,danilo,1991
4207,145707,2024.0,NaN,Danilo,Danilo,2025,614,danilo,Brazil,Bicas,...,https://www.transfermarkt.co.uk/danilo/profil/...,BRA1,Clube de Regatas do Flamengo,25000000.0,2024-10-18,7000000.0,BRA1,danilo,danilo,1991
16744,808509,2022.0,NaN,Danilo,Danilo,2025,537,danilo,Brazil,Salvador,...,https://www.transfermarkt.co.uk/danilo/profil/...,BRA1,S. A. F. Botafogo,28000000.0,2023-06-20,28000000.0,BRA1,danilo,danilo,2001
16745,808509,2023.0,NaN,Danilo,Danilo,2025,537,danilo,Brazil,Salvador,...,https://www.transfermarkt.co.uk/danilo/profil/...,BRA1,S. A. F. Botafogo,28000000.0,2024-05-27,28000000.0,BRA1,danilo,danilo,2001


In [29]:
from fuzzywuzzy import fuzz, process
import pandas as pd

# Initialisation des listes pour séparer les index
mapping_keys = df_mapping['join_key'].dropna().tolist()
indices_caches = []
indices_vrais_absents = []

print("Analyse des lignes en cours...")

# Classement ligne par ligne de still_missing
for idx, row in still_missing.iterrows():
    player_clean = str(row['player']).lower().strip()
    
    # Si le joueur est déjà un match exact parfait (normalement 0 ici)
    if player_clean in mapping_keys:
        indices_caches.append(idx)
        continue
        
    # Test fuzzy de contrôle (score à 75)
    best_match = process.extractOne(player_clean, mapping_keys, scorer=fuzz.token_set_ratio)
    
    if best_match and best_match[1] >= 75:
        indices_caches.append(idx)
    else:
        indices_vrais_absents.append(idx)

# Création des deux DataFrames cibles
df_caches = still_missing.loc[indices_caches].copy()
df_vrais_absents = still_missing.loc[indices_vrais_absents].copy()

# Colonnes d'affichage standard pour l'inspection
cols_affichage = ['player', 'team', 'season', 'league']


print(f"\nCas 1 : cachés par l'orthographe")
# On trie par équipe et joueur pour que ce soit lisible
print(df_caches[cols_affichage].sort_values(by=['team', 'player']).head(30).to_string(index=False))


print(f"\nCas 2 : vrais absents")
print(df_vrais_absents[cols_affichage].sort_values(by=['team', 'player']).head(30).to_string(index=False))

Analyse des lignes en cours...

Cas 1 : cachés par l'orthographe
             player            team  season             league
    Lander Pinillos          Alavés    2526        ESP-La Liga
         Marc Tenas          Alavés    2122        ESP-La Liga
   Youssef Lekhedim          Alavés    2526        ESP-La Liga
    Marciano Tchami         Almería    2324        ESP-La Liga
         Dan Sinaté          Angers    2526        FRA-Ligue 1
       Marius Louer          Angers    2526        FRA-Ligue 1
 Mohamed Amine Sbai          Angers    2526        FRA-Ligue 1
         Oumar Pona          Angers    2526        FRA-Ligue 1
         Max Dowman         Arsenal    2526 ENG-Premier League
    George Hemmings     Aston Villa    2526 ENG-Premier League
  Kosta Nedeljković     Aston Villa    2425 ENG-Premier League
 Lorenzo Bernasconi        Atalanta    2526        ITA-Serie A
       Asier Hierro   Athletic Club    2526        ESP-La Liga
      Antonio Gomis Atlético Madrid    2223        ES

In [26]:
df_caches

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
1,ENG-Premier League,2223,Brighton,Cameron Peupion,AUS,MF,19,2002.0,1,0,...,NaN,0,0.053613,0.000000,0.053613,0.000000,0.000000,cameron peupion,2002.0,2022
2,ENG-Premier League,2223,Manchester City,Shea Charles,NIR,"DF,MF",18,2003.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.129427,0.129427,shea charles,2003.0,2022
4,ENG-Premier League,2223,Tottenham Hotspur,George Abbott,ENG,MF,16,2005.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,george abbott,2005.0,2022
6,ENG-Premier League,2324,Bournemouth,Dominic Sadi,ENG,MF,19,2003.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.000000,0.000000,dominic sadi,2003.0,2023
9,ENG-Premier League,2324,Everton,Lewis Warrington,ENG,MF,20,2002.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.000000,0.000000,lewis warrington,2002.0,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428,ITA-Serie A,2526,Pisa,Samuele Angori,ITA,MF,22-209,2003.0,32,28,...,NaN,0,1.318561,3.855747,1.318561,8.059616,5.591388,samuele angori,2003.0,2025
430,ITA-Serie A,2526,Roma,Alessandro Romano,SUI,MF,19-321,2006.0,2,0,...,NaN,0,0.000000,0.000000,0.000000,0.533233,0.533233,alessandro romano,2006.0,2025
432,ITA-Serie A,2526,Sassuolo,Edoardo Iannoni,ITA,MF,25-023,2001.0,15,1,...,NaN,0,0.757199,0.000000,0.757199,0.894265,0.137066,edoardo iannoni,2001.0,2025
434,ITA-Serie A,2526,Sassuolo,Pedro Felipe,BRA,DF,21-346,2004.0,1,0,...,NaN,0,0.000000,0.374884,0.000000,0.883511,0.883511,pedro felipe,2004.0,2025


In [27]:
df_vrais_absents

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
0,ENG-Premier League,2021,Manchester Utd,Will Fish,ENG,DF,17,2003.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,will fish,2003.0,2020
3,ENG-Premier League,2223,Southampton,Kami Doyle,ENG,MF,16,2005.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,kami doyle,2005.0,2022
5,ENG-Premier League,2324,Aston Villa,Finley Munroe,ENG,DF,18,2005.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.000000,0.000000,finley munroe,2005.0,2023
7,ENG-Premier League,2324,Brighton,Mark O'Mahony,IRL,FW,18,2005.0,3,1,...,NaN,0,NaN,NaN,NaN,NaN,NaN,mark o mahony,2005.0,2023
8,ENG-Premier League,2324,Chelsea,Jimi Tauriainen,ENG,FW,19,2004.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.000000,0.000000,jimi tauriainen,2004.0,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
431,ITA-Serie A,2526,Roma,Jan Ziółkowski,POL,DF,20-333,2005.0,15,5,...,NaN,0,NaN,NaN,NaN,NaN,NaN,jan ziokowski,2005.0,2025
433,ITA-Serie A,2526,Sassuolo,Laurs Skjellerup,DEN,FW,23-265,2002.0,1,0,...,NaN,0,0.015538,0.000000,0.015538,0.015538,0.000000,laurs skjellerup,2002.0,2025
435,ITA-Serie A,2526,Sassuolo,Tarik Muharemovic,BIH,DF,23-065,2003.0,30,30,...,NaN,2,2.834149,0.927684,2.834149,6.279779,6.156958,tarik muharemovic,2003.0,2025
437,ITA-Serie A,2526,Udinese,Branimir Mlacic,CRO,DF,19-053,2007.0,5,1,...,NaN,0,0.000000,0.053524,0.000000,0.041268,0.041268,branimir mlacic,2007.0,2025
